In [ ]:
def _add_interaction_features(df: pd.DataFrame) -> pd.DataFrame:
    """Physics-motivated interaction terms for electricity price forecasting.

    Based on insights from Lago et al. (2021) "Forecasting day-ahead electricity
    prices: A review of state-of-the-art algorithms" and Ziel & Weron (2018)
    "Day-ahead electricity price forecasting with high-dimensional structures".
    Cross-feature products expose non-linear market dynamics that additive
    decision-tree splits cannot represent in a single feature independently.
    """
    df = df.copy()

    # Supply-stress × lagged price: when capacity headroom is tight AND recent
    # prices were high, forward prices spike non-linearly.
    if "supply_stress" in df.columns and "price_lag_1" in df.columns:
        df["supply_stress_x_price_l1"] = (
            df["supply_stress"].fillna(0.0) * df["price_lag_1"].fillna(0.0)
        ).astype(np.float32)

    # Predispatch RRP × supply-stress²: tighter than supply_stress alone because
    # it captures the dispatch-stack non-linearity at high loading.
    if "predispatch_rrp_1" in df.columns and "supply_stress" in df.columns:
        df["pd_rrp_x_supply_stress2"] = (
            df["predispatch_rrp_1"].fillna(0.0)
            * df["supply_stress"].fillna(0.0) ** 2
        ).astype(np.float32)

    # Coal outage × high-demand flag: demand-side amplification when thermal
    # capacity is absent raises prices faster than either alone.
    if "coal_outage_mw" in df.columns and "demand_lag_1" in df.columns:
        demand_75 = df["demand_lag_1"].quantile(0.75)
        df["coal_outage_x_high_demand"] = (
            df["coal_outage_mw"].fillna(0.0)
            * (df["demand_lag_1"] > demand_75).astype(np.float32)
        ).astype(np.float32)

    # Hot-day peak: temperature during afternoon peak is the primary driver of
    # air-conditioning load in NEM regions; interaction reveals the critical
    # temperature threshold behaviour.
    temp_col = next(
        (c for c in ("temp_sydney", "temp_brisbane", "temp_melbourne", "temp_adelaide")
         if c in df.columns),
        None,
    )
    if temp_col is not None and "is_peak" in df.columns:
        df["hot_day_peak"] = (
            np.maximum(df[temp_col].fillna(df[temp_col].median()) - 25.0, 0.0)
            * df["is_peak"]
        ).astype(np.float32)

    # Cooling-degree-days × demand lag: represents load-weighted temperature.
    if "cooling_deg_day" in df.columns and "demand_lag_1" in df.columns:
        df["cdd_x_demand"] = (
            df["cooling_deg_day"].fillna(0.0) * df["demand_lag_1"].fillna(0.0)
        ).astype(np.float32)

    # Spike-history × predispatch forecast: recent spike count amplified by
    # a high predispatch RRP is a strong forward-spike signal.
    if "spike_count_24h" in df.columns and "predispatch_rrp_1" in df.columns:
        df["spike_history_x_pd_forecast"] = (
            df["spike_count_24h"].fillna(0.0)
            * np.log1p(np.maximum(df["predispatch_rrp_1"].fillna(0.0), 0.0))
        ).astype(np.float32)

    # Forward-squeeze: predispatch RRP minus lagged spot; positive squeeze
    # (forward > spot) signals an anticipated tightening.
    if "predispatch_rrp_1" in df.columns and "price_lag_1" in df.columns:
        df["forward_squeeze"] = (
            df["predispatch_rrp_1"].fillna(0.0) - df["price_lag_1"].fillna(0.0)
        ).astype(np.float32)

    return df